# Clase 075 — Regresión logística binaria y softmax

La regresión logística convierte un score lineal en probabilidad con la **sigmoide** y se entrena minimizando **log-loss**. La generalizamos a multiclase con **softmax** y diagnosticamos la **calibración** de las probabilidades.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

Nota: en scikit-learn ≥ 1.7 el parámetro `multi_class` fue eliminado; la logística multiclase es **softmax (multinomial) por default** y para one-vs-rest se usa `OneVsRestClassifier`.

## 1. Logística binaria sobre breast cancer

Entrenamos `LogisticRegression` (escalada) y reportamos accuracy, log-loss y matriz de confusión. Interpretamos los coeficientes de mayor peso como *odds-ratio* $\exp(w)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix

data = load_breast_cancer()
Xc, yc = data.data, data.target
Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
proba = clf.predict_proba(Xte)[:, 1]

print('accuracy:', round(accuracy_score(yte, clf.predict(Xte)), 4))
print('log-loss:', round(log_loss(yte, proba), 4))
print('matriz de confusión:\n', confusion_matrix(yte, clf.predict(Xte)))
coef = clf.named_steps['logisticregression'].coef_[0]
for i in np.argsort(np.abs(coef))[::-1][:5]:
    print(f'{data.feature_names[i][:22]:22s} w={coef[i]:+.3f}  odds-ratio exp(w)={np.exp(coef[i]):.3f}')

## 2. Frontera de decisión y efecto de `C`

Con 2 features de iris (dos clases), graficamos la frontera lineal. `C` es la inversa de la regularización: `C` chico ⇒ frontera más rígida, `C` grande ⇒ más flexible.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
mask = iris.target < 2
X2, y2 = iris.data[mask][:, :2], iris.target[mask]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, C in zip(axes, [0.01, 100]):
    model = LogisticRegression(C=C).fit(X2, y2)
    xx, yy = np.meshgrid(np.linspace(X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5, 200),
                         np.linspace(X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap='coolwarm')
    ax.scatter(X2[:, 0], X2[:, 1], c=y2, cmap='coolwarm', edgecolor='k', s=20)
    ax.set_title(f'C = {C}'); ax.set_xlabel(iris.feature_names[0]); ax.set_ylabel(iris.feature_names[1])
plt.tight_layout(); plt.show()

## 3. Softmax vs one-vs-rest

Sobre iris (3 clases) comparamos la logística **multinomial** (softmax, default) contra `OneVsRestClassifier`. Verificamos que las probabilidades softmax de cada muestra suman 1.

In [ ]:
from sklearn.multiclass import OneVsRestClassifier

Xi, yi = iris.data, iris.target
Xtr, Xte, ytr, yte = train_test_split(Xi, yi, test_size=0.3, random_state=42, stratify=yi)
soft = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)   # softmax
ovr = make_pipeline(StandardScaler(),
                    OneVsRestClassifier(LogisticRegression(max_iter=1000))).fit(Xtr, ytr)

print('softmax -> acc:', round(accuracy_score(yte, soft.predict(Xte)), 4),
      '| log-loss:', round(log_loss(yte, soft.predict_proba(Xte)), 4))
print('OvR     -> acc:', round(accuracy_score(yte, ovr.predict(Xte)), 4),
      '| log-loss:', round(log_loss(yte, ovr.predict_proba(Xte)), 4))
p3 = soft.predict_proba(Xte[:3])
print('predict_proba de 3 muestras:\n', p3.round(3))
assert np.allclose(p3.sum(1), 1.0)
print('OK: las probabilidades softmax suman 1')

## 4. Reliability diagram de un RandomForest

Sobre un dataset desbalanceado, un RandomForest suele estar **mal calibrado**. Comparamos su `calibration_curve` con la diagonal y reportamos el Brier score.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

Xd, yd = make_classification(n_samples=6000, n_features=20, n_informative=5,
                             weights=[0.9, 0.1], random_state=42)
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.3, random_state=42, stratify=yd)
rf = RandomForestClassifier(n_estimators=60, random_state=42, n_jobs=1).fit(Xtr, ytr)
p_rf = rf.predict_proba(Xte)[:, 1]
frac, mean_pred = calibration_curve(yte, p_rf, n_bins=10)
brier_rf = brier_score_loss(yte, p_rf)
print('Brier score RF crudo:', round(brier_rf, 4))

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], 'k--', label='perfecto')
ax.plot(mean_pred, frac, 's-', label='RF crudo')
ax.set_xlabel('score medio'); ax.set_ylabel('fracción de positivos'); ax.legend()
ax.set_title('Reliability diagram — RandomForest')
plt.tight_layout(); plt.show()

## 5. Calibrar con `CalibratedClassifierCV`

Envolvemos el RF con calibración **Platt** (`sigmoid`) e **isotonic** y comparamos los tres reliability diagrams y sus Brier scores. La calibración no debe empeorar el Brier.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

base = lambda: RandomForestClassifier(n_estimators=60, random_state=42, n_jobs=1)
cal_sig = CalibratedClassifierCV(base(), method='sigmoid', cv=5).fit(Xtr, ytr)
cal_iso = CalibratedClassifierCV(base(), method='isotonic', cv=5).fit(Xtr, ytr)
p_sig, p_iso = cal_sig.predict_proba(Xte)[:, 1], cal_iso.predict_proba(Xte)[:, 1]
brier_sig, brier_iso = brier_score_loss(yte, p_sig), brier_score_loss(yte, p_iso)
print('Brier RF crudo  :', round(brier_rf, 4))
print('Brier +Platt    :', round(brier_sig, 4))
print('Brier +isotonic :', round(brier_iso, 4))

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], 'k--', label='perfecto')
for p, name in [(p_rf, 'RF crudo'), (p_sig, '+Platt'), (p_iso, '+isotonic')]:
    f, mp = calibration_curve(yte, p, n_bins=10)
    ax.plot(mp, f, 's-', label=name)
ax.legend(); ax.set_xlabel('score medio'); ax.set_ylabel('fracción de positivos')
ax.set_title('Calibración: RF crudo vs Platt vs isotonic')
plt.tight_layout(); plt.show()
assert min(brier_sig, brier_iso) <= brier_rf + 1e-6
print('OK: la calibración no empeora el Brier score')

## Ejercicios

1. **Odds-ratio.** Del ejercicio 1, tomá el coeficiente de mayor $|w|$ e interpretá su $\exp(w)$: ¿cuánto multiplica las odds de la clase positiva un aumento de una desviación estándar?
2. **`C` y frontera.** Barré `C ∈ {0.001, 0.1, 1, 100}` en la frontera de decisión y describí cómo cambia la rigidez.
3. **Softmax suma 1.** Verificá que las filas de `predict_proba` del modelo multinomial suman 1, pero las del OvR renormalizado no necesariamente reflejan un softmax real.
4. **Platt vs isotonic.** Repetí la calibración con distintos tamaños de dataset y decidí, según el Brier, cuál método conviene en cada régimen de datos.

## Conclusiones

- La **sigmoide** mapea el score lineal a $(0,1)$; la logística se entrena con **log-loss** (convexa) y no con MSE.
- **Softmax** generaliza la sigmoide a $K$ clases con probabilidades que suman 1; es el default multiclase en sklearn.
- **Accuracy no basta**: log-loss y Brier score miden la calidad de las **probabilidades**, no solo del argmax.
- Modelos como RandomForest suelen estar mal calibrados; `CalibratedClassifierCV` (Platt o isotonic) acerca el reliability diagram a la diagonal sin cambiar el ranking.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios de la seccion 🧪 **Ejercicios** del README. Cada bloque es autocontenido, se ejecuta **sin internet** y en pocos segundos. Intenta resolver cada ejercicio por tu cuenta antes de mirar la solucion.

### Ejercicio 1 — Logistica binaria sobre breast cancer
Accuracy, log-loss, matriz de confusion y odds-ratio de los 5 pesos mas fuertes. Reconstruimos el pipeline (el notebook reutiliza `Xte`/`clf` con otros datasets).

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix
bc = load_breast_cancer()
Xbctr, Xbcte, ybctr, ybcte = train_test_split(bc.data, bc.target, test_size=0.2,
                                              random_state=42, stratify=bc.target)
lrbc = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xbctr, ybctr)
proba_bc = lrbc.predict_proba(Xbcte)[:, 1]
pred = lrbc.predict(Xbcte)
print('accuracy:', round(accuracy_score(ybcte, pred), 4), '| log-loss:', round(log_loss(ybcte, proba_bc), 4))
print('confusion:'); print(confusion_matrix(ybcte, pred))
w = lrbc.named_steps['logisticregression'].coef_[0]
for i in np.argsort(np.abs(w))[::-1][:5]:
    print(f'{bc.feature_names[i][:24]:24s} w={w[i]:+.3f} odds-ratio={np.exp(w[i]):.2f}')
print('odds-ratio>1: subir esa feature (estandarizada) aumenta el odds de la clase 1.')

### Ejercicio 2 — Frontera de decision variando `C`
2 features de iris, 2 clases. `C` alto = frontera mas rigida.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
iris = load_iris()
mask = iris.target < 2
Xi = iris.data[mask][:, 2:4]; yi = iris.target[mask]
xx, yy = np.meshgrid(np.linspace(Xi[:, 0].min() - .5, Xi[:, 0].max() + .5, 200),
                     np.linspace(Xi[:, 1].min() - .5, Xi[:, 1].max() + .5, 200))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, C in zip(axes, [0.01, 1, 100]):
    mdl = make_pipeline(StandardScaler(), LogisticRegression(C=C)).fit(Xi, yi)
    Z = mdl.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap='coolwarm')
    ax.scatter(Xi[:, 0], Xi[:, 1], c=yi, cmap='coolwarm', edgecolor='k', s=18)
    ax.set_title(f'C={C}')
plt.tight_layout(); plt.show()

### Ejercicio 3 — Softmax vs OvR sobre iris (3 clases)
Comparamos log-loss y accuracy; las probas de softmax suman 1.

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss
Xi, yi = iris.data, iris.target
Xtr3, Xte3, ytr3, yte3 = train_test_split(Xi, yi, test_size=0.3, stratify=yi, random_state=42)
soft = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr3, ytr3)  # multinomial por defecto
ovr = make_pipeline(StandardScaler(), OneVsRestClassifier(LogisticRegression(max_iter=1000))).fit(Xtr3, ytr3)
for name, mdl in [('softmax', soft), ('ovr', ovr)]:
    P = mdl.predict_proba(Xte3)
    print(f'{name:8s} acc={accuracy_score(yte3, mdl.predict(Xte3)):.3f} log_loss={log_loss(yte3, P):.3f}')
print('proba de 3 muestras (softmax) suma 1:', soft.predict_proba(Xte3)[:3].sum(1).round(3))

### Ejercicio 4 — Reliability diagram de un RandomForest
Sobre un dataset desbalanceado; reportamos Brier score.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
Xb, yb = make_classification(n_samples=5000, n_features=20, weights=[0.9, 0.1], random_state=42)
Xbtr, Xbte, ybtr, ybte = train_test_split(Xb, yb, test_size=0.3, stratify=yb, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xbtr, ybtr)
pb = rf.predict_proba(Xbte)[:, 1]
frac, mean_pred = calibration_curve(ybte, pb, n_bins=10)
plt.figure(figsize=(5, 5)); plt.plot([0, 1], [0, 1], 'k--', label='perfecto')
plt.plot(mean_pred, frac, 'o-', label='RF'); plt.legend()
plt.xlabel('proba media'); plt.ylabel('frac positivos'); plt.title('RF reliability')
plt.tight_layout(); plt.show()
print('Brier RF:', round(brier_score_loss(ybte, pb), 4))

### Ejercicio 5 — Calibrar con `CalibratedClassifierCV`
Platt (sigmoid) vs isotonic; menor Brier = mejor calibrado.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
res = {'RF crudo': pb}
for method in ['sigmoid', 'isotonic']:
    cal = CalibratedClassifierCV(RandomForestClassifier(n_estimators=100, random_state=42),
                                 method=method, cv=5).fit(Xbtr, ybtr)
    res[method] = cal.predict_proba(Xbte)[:, 1]
fig, ax = plt.subplots(figsize=(5, 5)); ax.plot([0, 1], [0, 1], 'k--')
for name, p in res.items():
    f, mp = calibration_curve(ybte, p, n_bins=10)
    ax.plot(mp, f, 'o-', label=f'{name} (Brier {brier_score_loss(ybte, p):.3f})')
ax.legend(); ax.set_title('crudo vs Platt vs isotonic'); plt.tight_layout(); plt.show()